# Class 1: What is NLP?
## Bengali Text Processing → TF-IDF → First Binary Toxicity Classifier

**Course project:** Bengali NLP Safety Challenge 2026  
**Primary target for this class:** `is_toxic`

> **Important:** We deliberately keep the model simple today.  
> Class 2 will extend this into multi-label learning and Kaggle submission.

## roadmap

```text
"What is NLP?"
       ↓
Explore Bengali toxicity data
       ↓
Bengali preprocessing
       ↓
Text → numbers
       ↓
TF-IDF
       ↓
Logistic Regression
       ↓
Binary target: is_toxic
       ↓
Evaluate + inspect mistakes
       ↓
First real NLP classifier
```

# 1. What is NLP?

**Natural Language Processing (NLP)** is the field of AI that helps computers work with human language.

Examples:

- spam detection
- sentiment analysis
- machine translation
- search
- chatbots
- question answering
- toxicity and safety classification

For this course, our question is:

> **Can a machine learn whether a Bengali text is toxic?**

A computer does not naturally understand a sentence such as:

```text
তুমি খুব ভালো মানুষ
```

A machine-learning algorithm typically expects numbers.

So our first pipeline is:

```text
Bengali text
    ↓
preprocessing
    ↓
numerical representation
    ↓
machine-learning model
    ↓
prediction
```

## NLP task taxonomy

A few common NLP tasks:

| Task | Input | Output |
|---|---|---|
| Sentiment analysis | review text | positive / negative |
| Spam detection | message | spam / not spam |
| Topic classification | article | politics / sports / tech |
| Named entity recognition | sentence | person / place / organization |
| Translation | Bengali text | English text |
| Toxicity detection | Bengali text | toxic / non-toxic |

### Our Class 1 problem

We begin with a **binary classification** problem:

\[
y \in \{0,1\}
\]

where:

- `0` = non-toxic
- `1` = toxic

# 2. Setup

We will use:

- `pandas` for data handling
- `scikit-learn` for TF-IDF and machine learning
- `matplotlib` for basic plots

The notebook is configured for the project directory supplied by the instructor.  
If you run it elsewhere, change `DATA_DIR`.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

DATA_DIR = Path("/home/mithon-hossain/projects/toxic/final_dataset")
TRAIN_PATH = DATA_DIR / "train.csv"

print("Looking for:", TRAIN_PATH)

: 

## Load the training dataset

We will first inspect the available columns rather than assuming too much.

The expected Class 1 target is:

```text
is_toxic
```

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)

print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

train_df.head()

## Find the text column safely

Different dataset versions sometimes use names such as `text`, `comment_text`, `content`, or `sentence`.

This helper tries common names first. If none are found, it searches object/string columns.

In [ ]:
TEXT_CANDIDATES = [
    "text",
    "comment_text",
    "comment",
    "content",
    "sentence",
    "bengali_text",
]

def infer_text_column(df):
    for col in TEXT_CANDIDATES:
        if col in df.columns:
            return col

    object_cols = [
        c for c in df.columns
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
    ]

    if len(object_cols) == 1:
        return object_cols[0]

    raise ValueError(
        "Could not automatically identify the text column. "
        f"String-like columns found: {object_cols}"
    )

TEXT_COL = infer_text_column(train_df)
TARGET_COL = "is_toxic"

assert TARGET_COL in train_df.columns, (
    f"Expected target '{TARGET_COL}' was not found. "
    f"Available columns: {train_df.columns.tolist()}"
)

print("Text column :", TEXT_COL)
print("Target column:", TARGET_COL)

# 3. First look at the data

Before preprocessing or modeling, always **look at the data**.

Questions to ask:

1. How many examples do we have?
2. Are there missing texts?
3. Are there duplicate texts?
4. How balanced is the target?
5. What does Bengali toxicity look like in real examples?

In [ ]:
print("Rows:", len(train_df))
print("Missing text:", train_df[TEXT_COL].isna().sum())
print("Missing target:", train_df[TARGET_COL].isna().sum())
print("Duplicate texts:", train_df[TEXT_COL].duplicated().sum())

display(
    train_df[[TEXT_COL, TARGET_COL]]
    .sample(min(8, len(train_df)), random_state=RANDOM_STATE)
)

## Target distribution

A classifier can behave very differently when one class is much more common than the other.

Today we only *observe* the distribution.  
We will study class imbalance more deeply in Class 4.

In [ ]:
target_counts = train_df[TARGET_COL].value_counts().sort_index()
target_pct = train_df[TARGET_COL].value_counts(normalize=True).sort_index() * 100

distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct.round(2)
})
distribution

In [ ]:
ax = target_counts.plot(kind="bar", figsize=(6, 4))
ax.set_title("Distribution of is_toxic")
ax.set_xlabel("Class")
ax.set_ylabel("Number of examples")
ax.set_xticklabels(["0 = Non-toxic", "1 = Toxic"], rotation=0)
plt.show()

# 4. Bengali text is messy, and that is normal

Real Bengali text can contain:

- বাংলা Unicode characters
- English words
- Banglish or transliteration
- punctuation
- repeated punctuation
- digits
- emojis
- URLs
- usernames
- extra spaces
- spelling variation

Example:

```text
এইটা কি বললা!!! 😡
```

A common beginner mistake is to **clean too aggressively**.

For toxicity detection, punctuation, emojis, repeated symbols, and spelling patterns may contain useful information.

So our Class 1 preprocessing will be intentionally conservative.

## Preprocessing principles for this task

We will:

1. convert missing values to empty strings
2. normalize Unicode
3. lowercase English characters
4. replace URLs with a token
5. replace user mentions with a token
6. normalize repeated whitespace

We will **not**:

- remove Bengali words
- stem Bengali words
- remove every punctuation mark
- remove emojis
- translate the text
- remove so-called stopwords automatically

Why?

Because some of those signals may help detect toxicity.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")

def preprocess_bengali_text(text):
    """
    Lightweight preprocessing suitable for a first Bengali toxicity baseline.

    We intentionally preserve most lexical and stylistic information.
    """
    text = "" if pd.isna(text) else str(text)

    # Normalize Unicode representation
    text = unicodedata.normalize("NFC", text)

    # Lowercase English letters without affecting Bengali characters
    text = text.lower()

    # Replace a few noisy structures with stable tokens
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)

    # Normalize whitespace
    text = SPACE_RE.sub(" ", text).strip()

    return text

## Inspect preprocessing before applying it everywhere

Never trust a preprocessing function blindly.

Look at examples before and after.

In [ ]:
examples = train_df[[TEXT_COL]].sample(
    min(10, len(train_df)),
    random_state=RANDOM_STATE
).copy()

examples["processed_text"] = examples[TEXT_COL].apply(preprocess_bengali_text)

examples

### Discussion

Ask the class:

- What information did we remove?
- What information did we preserve?
- Should `!!!` be removed in a toxicity task?
- Could emojis carry safety information?
- Would aggressive stemming always help Bengali?

There is no universal preprocessing recipe.

**Preprocessing should match the task.**

# 5. How do we turn text into numbers?

Suppose our vocabulary contains:

```text
তুমি
ভালো
খারাপ
মানুষ
```

A simple **Bag of Words** representation records whether, or how often, each word occurs.

Example:

```text
"তুমি ভালো মানুষ"
```

might become:

```text
[1, 1, 0, 1]
```

The problem is that very common words can dominate.

TF-IDF gives more importance to terms that are informative for particular documents.

# 6. TF-IDF

**TF-IDF** means:

> Term Frequency × Inverse Document Frequency

A simplified intuition:

### Term Frequency

How often does a term occur in this document?

\[
TF(t,d)
\]

### Inverse Document Frequency

How rare is the term across the entire collection?

\[
IDF(t) = \log\left(\frac{N}{df(t)}\right)
\]

### TF-IDF

\[
TFIDF(t,d) = TF(t,d)\times IDF(t)
\]

Words that occur everywhere get less weight.

Words that are more distinctive can receive larger weights.

## A tiny TF-IDF demonstration

This artificial example is only for understanding the representation.

In [ ]:
toy_texts = [
    "তুমি ভালো মানুষ",
    "তুমি খারাপ মানুষ",
    "খুব ভালো",
]

toy_vectorizer = TfidfVectorizer()
toy_matrix = toy_vectorizer.fit_transform(toy_texts)

toy_tfidf = pd.DataFrame(
    toy_matrix.toarray(),
    columns=toy_vectorizer.get_feature_names_out()
)

toy_tfidf.round(3)

### Important observation

TF-IDF does **not** understand language like a human.

It represents statistical patterns in tokens.

Still, TF-IDF is often a surprisingly strong baseline for text classification.

# 7. Prepare the binary classification dataset

We now create:

```text
X = Bengali text
y = is_toxic
```

Then split the labeled examples into training and validation sets.

Why do we need a validation set?

Because evaluating on the same data used for training gives an unrealistically optimistic result.

In [ ]:
model_df = train_df[[TEXT_COL, TARGET_COL]].copy()

model_df = model_df.dropna(subset=[TARGET_COL])
model_df[TEXT_COL] = model_df[TEXT_COL].fillna("")
model_df["processed_text"] = model_df[TEXT_COL].apply(preprocess_bengali_text)

X = model_df["processed_text"]
y = model_df[TARGET_COL].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training examples  :", len(X_train))
print("Validation examples:", len(X_valid))
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())

# 8. Build TF-IDF features

For Class 1 we use **word-level TF-IDF**.

Parameters:

- `ngram_range=(1, 2)`  
  use single words and two-word sequences

- `min_df=2`  
  ignore tokens appearing in only one training example

- `max_features=50_000`  
  put an upper bound on vocabulary size

Later, Class 2 will combine **word + character TF-IDF**.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_valid)

print("Training matrix shape :", X_train_tfidf.shape)
print("Validation matrix shape:", X_valid_tfidf.shape)
print("Vocabulary size        :", len(tfidf.vocabulary_))

## Why `fit_transform` on training data but only `transform` on validation data?

This is very important.

```text
TRAIN
  ↓
learn vocabulary + IDF weights
  ↓
transform

VALIDATION
  ↓
use the already learned vocabulary + weights
  ↓
transform
```

If we fit TF-IDF using validation data too, information leaks from validation into training.

That is called **data leakage**.

# 9. Train our first classifier

We will use **Logistic Regression**.

Despite its name, Logistic Regression is commonly used for classification.

For binary classification, the model estimates:

\[
P(y=1 \mid x)
\]

If the probability is above a threshold, usually `0.5`, we predict class `1`.

In [ ]:
clf = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

clf.fit(X_train_tfidf, y_train)

valid_pred = clf.predict(X_valid_tfidf)
valid_prob = clf.predict_proba(X_valid_tfidf)[:, 1]

print("Model trained successfully.")

# 10. Evaluate the first model

Today we will calculate several familiar metrics.

Class 2 will study **F1 and Macro-F1** properly because they are crucial for the competition.

In [ ]:
accuracy = accuracy_score(y_valid, valid_pred)
precision = precision_score(y_valid, valid_pred, zero_division=0)
recall = recall_score(y_valid, valid_pred, zero_division=0)
f1 = f1_score(y_valid, valid_pred, zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

In [ ]:
print(classification_report(
    y_valid,
    valid_pred,
    target_names=["Non-toxic", "Toxic"],
    digits=4,
    zero_division=0,
))

## Confusion matrix

A confusion matrix shows four possibilities:

| | Predicted non-toxic | Predicted toxic |
|---|---:|---:|
| **Actually non-toxic** | correct | false positive |
| **Actually toxic** | false negative | correct |

For a safety system, false negatives can be especially interesting:

> Toxic content that the model failed to detect.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_pred,
    display_labels=["Non-toxic", "Toxic"],
)
plt.title("Validation Confusion Matrix")
plt.show()

# 11. Look at probabilities, not only labels

A classifier can be:

```text
0.51 toxic
```

or:

```text
0.99 toxic
```

Both produce class `1` at threshold `0.5`, but the model confidence is very different.

Threshold selection becomes important later in the course.

In [ ]:
validation_results = pd.DataFrame({
    "text": X_valid.values,
    "true_label": y_valid.values,
    "predicted_label": valid_pred,
    "toxic_probability": valid_prob,
})

validation_results.head()

# 12. Error analysis

A model score tells us **how much** the model is wrong.

Error analysis helps us understand **why** it is wrong.

We will inspect:

1. false positives
2. false negatives
3. uncertain predictions

In [ ]:
false_positives = validation_results[
    (validation_results["true_label"] == 0) &
    (validation_results["predicted_label"] == 1)
].sort_values("toxic_probability", ascending=False)

false_negatives = validation_results[
    (validation_results["true_label"] == 1) &
    (validation_results["predicted_label"] == 0)
].sort_values("toxic_probability", ascending=True)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

In [ ]:
display(false_positives.head(10))

In [ ]:
display(false_negatives.head(10))

### Class discussion

Look for possible patterns:

- sarcasm
- implicit toxicity
- uncommon spelling
- very short comments
- code-mixing
- words unseen during training
- profanity used in a non-toxic context
- toxicity without obvious profanity

This is where we begin to discover why simple lexical models have limitations.

## Most uncertain examples

Probabilities close to `0.5` are cases where the model is least certain.

In [ ]:
validation_results["uncertainty"] = (
    validation_results["toxic_probability"] - 0.5
).abs()

validation_results.sort_values("uncertainty").head(15)

# 13. What words influence the model?

For a linear model such as Logistic Regression, we can inspect feature coefficients.

Large positive coefficients push predictions toward **toxic**.

Large negative coefficients push predictions toward **non-toxic**.

> Caution: This shows statistical association, not human-like understanding.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefficients = clf.coef_[0]

top_toxic_idx = np.argsort(coefficients)[-20:][::-1]
top_nontoxic_idx = np.argsort(coefficients)[:20]

top_toxic_features = pd.DataFrame({
    "feature": feature_names[top_toxic_idx],
    "coefficient": coefficients[top_toxic_idx],
})

top_nontoxic_features = pd.DataFrame({
    "feature": feature_names[top_nontoxic_idx],
    "coefficient": coefficients[top_nontoxic_idx],
})

print("Features most associated with TOXIC predictions")
display(top_toxic_features)

print("Features most associated with NON-TOXIC predictions")
display(top_nontoxic_features)

# 14. Predict your own Bengali text

Now use the trained pipeline interactively.

In [ ]:
def predict_toxicity(text, threshold=0.5):
    cleaned = preprocess_bengali_text(text)
    features = tfidf.transform([cleaned])
    probability = clf.predict_proba(features)[0, 1]
    prediction = int(probability >= threshold)

    return {
        "text": text,
        "processed_text": cleaned,
        "toxic_probability": round(float(probability), 4),
        "prediction": prediction,
        "label": "TOXIC" if prediction == 1 else "NON-TOXIC",
    }

In [ ]:
# Replace these examples with your own Bengali sentences.
examples_to_test = [
    "তুমি অনেক ভালো কাজ করেছ",
    "এই কথাটা আমার একদম পছন্দ হয়নি",
]

for text in examples_to_test:
    print(predict_toxicity(text))

# 15. Why is this model useful?

Our first model already has a complete machine-learning pipeline:

```text
Raw Bengali text
      ↓
Light preprocessing
      ↓
TF-IDF
      ↓
Logistic Regression
      ↓
P(toxic)
      ↓
0 / 1 prediction
```

That is a real NLP classification system.

But it still has major limitations.

# 16. Limitations of TF-IDF

Consider:

```text
তুমি তো অনেক ভালো মানুষ!
```

Depending on context, this could be sincere or sarcastic.

TF-IDF mostly sees words and short token patterns.

It does not deeply model:

- sentence context
- long-distance relationships
- nuanced sarcasm
- meaning changes caused by context
- semantic similarity between unseen phrases

These limitations will motivate **Attention, Transformers, and BERT in Class 3**.

# 17. In-class exercises

## Exercise 1: Explore the data

Find:

1. five toxic examples
2. five non-toxic examples
3. the longest text
4. the shortest non-empty text

---

## Exercise 2: Character count

Create a new column:

```python
text_length
```

containing the number of characters in every text.

Compare average length for toxic vs non-toxic examples.

---

## Exercise 3: Change TF-IDF

Try:

```python
ngram_range=(1, 1)
```

and compare it with:

```python
ngram_range=(1, 2)
```

Does the validation score change?

---

## Exercise 4: Inspect mistakes

Choose:

- 3 false positives
- 3 false negatives

For each one, write one sentence explaining why TF-IDF might have struggled.

---

## Exercise 5: Test your own sentences

Write five Bengali sentences and run:

```python
predict_toxicity(...)
```

Try to create at least one example that fools the model.

# 18. Optional challenge: text length analysis

In [ ]:
analysis_df = model_df.copy()
analysis_df["text_length"] = analysis_df["processed_text"].str.len()

analysis_df.groupby(TARGET_COL)["text_length"].agg(
    ["count", "mean", "median", "min", "max"]
)

# 19. What did we learn?

Today we learned:

```text
Human language
     ↓
NLP task
     ↓
Bengali text
     ↓
Preprocessing
     ↓
TF-IDF numerical features
     ↓
Binary classifier
     ↓
is_toxic prediction
```

### Key ideas

- NLP connects human language with computational models.
- Text must be represented numerically before classical ML can use it.
- Preprocessing should be **task-aware**, not blindly aggressive.
- TF-IDF is a strong classical representation for text.
- Logistic Regression provides a simple but effective baseline.
- Validation is necessary to estimate generalization.
- Looking at mistakes is as important as looking at scores.

# 20. Bridge to Class 2

Today we asked only:

> **Is this text toxic?**

But Bengali safety is not one-dimensional.

A text can involve different kinds of harmful content, such as:

```text
implicit_toxicity
sarcasm
contains_profanity
threatening
identity_attack
sexual_content
```

One text may belong to **multiple categories at the same time**.

So next class:

```text
"How do we evaluate NLP?"
       ↓
Precision / Recall / F1
       ↓
Macro-F1
       ↓
Multi-label learning
       ↓
Word + Character TF-IDF
       ↓
7-target prediction
       ↓
First Kaggle submission
```

# Homework

Before Class 2:

1. Run the entire notebook.
2. Record your validation Accuracy and F1.
3. Find three false positives and three false negatives.
4. Write down one hypothesis for why each error occurred.
5. Experiment with one TF-IDF parameter.
6. Bring one Bengali example that the model predicts incorrectly.

### Reflection question

> If `is_toxic = 1`, does that tell us *what kind* of toxicity is present?

That question leads directly into multi-label classification.